In [0]:
%sql
use ecommerce_db;

In [0]:
%sql
-- performance optimization
--optimizing
OPTIMIZE silver_ecommerce;

path,metrics
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 1, 1, true, 0, 0, 1772084439162, 1772084440589, 8, 0, null, List(0, 0), null, 9, 9, 0, 0, null, null)"


In [0]:
%sql
-- z ordering

OPTIMIZE silver_ecommerce
ZORDER BY (CustomerID);

path,metrics
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, List(minCubeSize(107374182400), List(0, 0), List(1, 2643036), 0, List(0, 0), 0, null), null, 0, 0, 1, 1, false, 0, 0, 1772084468441, 1772084469182, 8, 0, null, List(0, 0), null, 9, 9, 0, 0, null, null)"


In [0]:
%sql
-- time travel

DESCRIBE HISTORY silver_ecommerce;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2026-02-23T16:47:01.000Z,72433057351155,adityachitta06@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(1654599041107359),7eefae33-f206-42a7-abfa-2f4f56de0c33,0223-163252-zpcna569-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 397924, numOutputBytes -> 2643036)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13


In [0]:
%sql
-- reading old versions in time travel

SELECT COUNT(*) 
FROM silver_ecommerce VERSION AS OF 0;

COUNT(*)
397924


In [0]:
# merging (simulating incremental data)
from pyspark.sql import Row

new_data = [
    Row(InvoiceNo="999999", StockCode="TEST1", Description="Test Product",
        Quantity=5, InvoiceDate="2011-12-10 10:00:00",
        UnitPrice=20.0, CustomerID=12345, Country="United Kingdom")
]

df_new = spark.createDataFrame(new_data)
display(df_new)

InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
999999,TEST1,Test Product,5,2011-12-10 10:00:00,20.0,12345,United Kingdom


In [0]:

%sql
--now merging it

MERGE INTO silver_ecommerce AS target
USING (
  SELECT *,
         Quantity * UnitPrice AS total_amount
  FROM VALUES
    ('999999','TEST1','Test Product',5,'2011-12-10 10:00:00',20.0,12345,'United Kingdom')
  AS t(InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country)
) AS source
ON target.InvoiceNo = source.InvoiceNo
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
1,0,0,1
